# Análisis de Datos (EDA) — Mi Spotify Wrapped DWH

**Universidad de Pamplona · Bases de Datos II · 2026-I**  
**Profesor:** Juan Alejandro Carrillo Jaimes  
**Integrantes:** Jhonatan Vera · Suley Suárez  
**Fecha:** Mayo 2026

## Configuración e Imports

In [ ]:
from dotenv import load_dotenv
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sqlalchemy import create_engine
import warnings

warnings.filterwarnings('ignore')
load_dotenv(dotenv_path='../backend/.env')

DATABASE_URL = os.getenv('DATABASE_URL')
assert DATABASE_URL, 'DATABASE_URL no encontrada en .env'

engine = create_engine(DATABASE_URL)
print('Conexión exitosa al DWH.')

# Estilo global de gráficos
plt.rcParams.update({
    'figure.facecolor': '#1a1a2e',
    'axes.facecolor':   '#16213e',
    'axes.edgecolor':   '#0f3460',
    'axes.labelcolor':  'white',
    'xtick.color':      'white',
    'ytick.color':      'white',
    'text.color':       'white',
    'grid.color':       '#0f3460',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
})
SPOTIFY_GREEN = '#1DB954'

---
## Paso 1 — Conexión y revisión de datos

In [ ]:
df_users   = pd.read_sql('SELECT * FROM dwh.dim_users',             engine)
df_artists = pd.read_sql('SELECT * FROM dwh.dim_artists',           engine)
df_tracks  = pd.read_sql('SELECT * FROM dwh.dim_tracks',            engine)
df_history = pd.read_sql('SELECT * FROM dwh.fact_listening_history', engine)

print(f'dim_users:              {df_users.shape}')
print(f'dim_artists:            {df_artists.shape}')
print(f'dim_tracks:             {df_tracks.shape}')
print(f'fact_listening_history: {df_history.shape}')

In [ ]:
print('=== fact_listening_history ===')
print(df_history.shape)
print(df_history.dtypes)
print('\n% nulos por columna:')
print((df_history.isnull().mean() * 100).round(2))
df_history.head(3)

In [ ]:
print('=== dim_artists ===')
print(df_artists.shape)
print(df_artists.dtypes)
print('\n% nulos por columna:')
print((df_artists.isnull().mean() * 100).round(2))
df_artists.head(3)

In [ ]:
print('=== dim_tracks ===')
print(df_tracks.shape)
print(df_tracks.dtypes)
print('\n% nulos por columna:')
print((df_tracks.isnull().mean() * 100).round(2))
df_tracks.head(3)

In [ ]:
print('=== dim_users ===')
print(df_users.shape)
print(df_users.dtypes)
print('\n% nulos por columna:')
print((df_users.isnull().mean() * 100).round(2))
df_users.head(3)

**Observaciones sobre la calidad de los datos:**

- `fact_listening_history` contiene el historial de reproducción real del usuario, con campos derivados como `hour_of_day` y `day_of_week` calculados en el ETL.
- `dim_artists.genres` es un array PostgreSQL; al cargarlo en pandas se convierte en lista Python. Los artistas con géneros vacíos pueden mostrar `['']` como sentinel del backfill.
- Los nulos en `popularity` e `image_url` de artistas corresponden a artistas creados como stub desde el historial antes del primer enrichment con Spotify API.
- No se detectan fechas fuera de rango en `played_at`.

---
## Paso 2 — Estadística Descriptiva

In [ ]:
df_artists[['popularity', 'followers_count']].describe().round(2)

**Interpretación dim_artists:**  
La popularidad promedio de los artistas escuchados ronda entre 40 y 70 (escala 0–100 de Spotify), lo que indica un perfil de oyente que mezcla artistas con exposición media-alta con algunos más de nicho. La gran dispersión en `followers_count` refleja que el catálogo incluye tanto artistas de culto regional como artistas con millones de seguidores.

In [ ]:
desc_tracks = df_tracks[['duration_ms', 'popularity']].describe().round(2)
desc_tracks['duration_min'] = (desc_tracks['duration_ms'] / 60000).round(2)
print(desc_tracks)

**Interpretación dim_tracks:**  
La duración promedio de las canciones es aproximadamente 3–4 minutos, coherente con el formato pop/urbano estándar. La popularidad de las canciones presenta una distribución variada: la mediana es inferior al promedio, lo que sugiere que unas pocas canciones muy virales elevan la media, mientras que la mayoría son de popularidad moderada.

---
## Pregunta 1 — ¿Cuáles son mis 10 artistas más escuchados?

In [ ]:
top_artists = pd.read_sql("""
    SELECT a.name AS artist,
           COUNT(*) AS plays
    FROM dwh.fact_listening_history h
    JOIN dwh.dim_artists a ON h.artist_id = a.artist_id
    GROUP BY a.name
    ORDER BY plays DESC
    LIMIT 10
""", engine)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(
    top_artists['artist'][::-1],
    top_artists['plays'][::-1],
    color=SPOTIFY_GREEN,
    edgecolor='none',
    height=0.65,
)
for bar in bars:
    ax.text(
        bar.get_width() + 0.3,
        bar.get_y() + bar.get_height() / 2,
        str(int(bar.get_width())),
        va='center', ha='left', fontsize=9, color='white'
    )
ax.set_xlabel('Reproducciones', fontsize=11)
ax.set_title('Top 10 Artistas Más Escuchados', fontsize=14, fontweight='bold', pad=15)
ax.grid(axis='x')
ax.spines[['top','right','left','bottom']].set_visible(False)
plt.tight_layout()
plt.savefig('p1_top_artists.png', dpi=150, bbox_inches='tight')
plt.show()

**Interpretación:**  
El artista con más reproducciones acumula notablemente más plays que el resto, lo que revela una escucha muy concentrada en un artista favorito. Los géneros cristianos y de adoración dominan el top, lo cual era esperado dado el perfil del usuario. Hay algún artista pop/reggaeton que aparece más de lo esperado, posiblemente por épocas de escucha más frecuente.

---
## Pregunta 2 — ¿A qué hora del día escucho más música?

In [ ]:
hourly = pd.read_sql("""
    SELECT hour_of_day, COUNT(*) AS plays
    FROM dwh.fact_listening_history
    GROUP BY hour_of_day
    ORDER BY hour_of_day
""", engine)

# Asegurar que estén las 24 horas aunque algunas tengan 0 reproducciones
all_hours = pd.DataFrame({'hour_of_day': range(24)})
hourly = all_hours.merge(hourly, on='hour_of_day', how='left').fillna(0)
hourly['plays'] = hourly['plays'].astype(int)

peak_hour = hourly.loc[hourly['plays'].idxmax(), 'hour_of_day']

fig, ax = plt.subplots(figsize=(12, 5))
colors = [SPOTIFY_GREEN if h == peak_hour else '#2d6a4f' for h in hourly['hour_of_day']]
ax.bar(hourly['hour_of_day'], hourly['plays'], color=colors, edgecolor='none', width=0.8)
ax.axvline(peak_hour, color='white', linestyle='--', linewidth=1.2, alpha=0.6, label=f'Hora pico: {int(peak_hour):02d}:00')
ax.set_xlabel('Hora del día (0 = medianoche)', fontsize=11)
ax.set_ylabel('Reproducciones', fontsize=11)
ax.set_title('Distribución de Reproducciones por Hora del Día', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(range(24))
ax.set_xticklabels([f'{h:02d}h' for h in range(24)], rotation=45, ha='right', fontsize=8)
ax.legend(fontsize=10)
ax.grid(axis='y')
ax.spines[['top','right','left','bottom']].set_visible(False)
plt.tight_layout()
plt.savefig('p2_hour_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Hora con más reproducciones: {int(peak_hour):02d}:00 ({hourly["plays"].max()} plays)')

**Interpretación:**  
El pico de escucha se concentra entre las horas de la tarde-noche, franjas que típicamente corresponden al regreso a casa o tiempo de estudio nocturno. La madrugada tiene actividad muy baja o nula, lo que confirma un patrón de escucha diurno. La franja de mediodía también muestra un pico menor, posiblemente asociado con el descanso del almuerzo.

---
## Pregunta 3 — ¿Qué tan popular es la música que escucho?

In [ ]:
pop_df = pd.read_sql("""
    SELECT t.popularity
    FROM dwh.fact_listening_history h
    JOIN dwh.dim_tracks t ON h.track_id = t.track_id
    WHERE t.popularity IS NOT NULL
""", engine)

mean_pop   = pop_df['popularity'].mean()
median_pop = pop_df['popularity'].median()
print(f'Media:   {mean_pop:.1f}')
print(f'Mediana: {median_pop:.1f}')
print(f'Std:     {pop_df["popularity"].std():.1f}')
print(pop_df['popularity'].describe().round(2))

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(pop_df['popularity'], bins=20, color=SPOTIFY_GREEN, edgecolor='#121212', alpha=0.85)
ax.axvline(mean_pop,   color='white',  linestyle='--', linewidth=1.5, label=f'Media: {mean_pop:.1f}')
ax.axvline(median_pop, color='yellow', linestyle=':',  linewidth=1.5, label=f'Mediana: {median_pop:.1f}')
ax.set_xlabel('Popularidad (0 = underground, 100 = viral)', fontsize=11)
ax.set_ylabel('Cantidad de reproducciones', fontsize=11)
ax.set_title('Distribución de Popularidad de las Canciones Escuchadas', fontsize=14, fontweight='bold', pad=15)
ax.legend(fontsize=10)
ax.grid(axis='y')
ax.spines[['top','right','left','bottom']].set_visible(False)
plt.tight_layout()
plt.savefig('p3_popularity_dist.png', dpi=150, bbox_inches='tight')
plt.show()

**Interpretación:**  
La distribución de popularidad muestra que el usuario no es puramente mainstream ni underground — hay reproducciones distribuidas a lo largo de todo el espectro. La concentración mayor en rangos medios (30–70) sugiere un perfil ecléctico que consume tanto artistas con exposición moderada como algunos más virales. El segmento de baja popularidad (0–30) corresponde principalmente a artistas de adoración y música cristiana latinoamericana con menor presencia global en Spotify.

---
## Pregunta 4 — ¿Qué géneros dominan mi historial?

In [ ]:
# UNNEST en SQL para desanidar el array de géneros de PostgreSQL
genres_df = pd.read_sql("""
    SELECT LOWER(TRIM(UNNEST(a.genres))) AS genre,
           COUNT(*) AS plays
    FROM dwh.fact_listening_history h
    JOIN dwh.dim_artists a ON h.artist_id = a.artist_id
    WHERE a.genres IS NOT NULL
      AND array_length(a.genres, 1) > 0
      AND a.genres != '{""}'::text[]
    GROUP BY genre
    HAVING LOWER(TRIM(UNNEST(a.genres))) != ''
    ORDER BY plays DESC
    LIMIT 15
""", engine)

# Alternativa Pandas si el SQL da problemas con HAVING + UNNEST
if genres_df.empty or len(genres_df) < 3:
    print('Usando método Pandas nativo como fallback...')
    merged = df_history.merge(df_artists[['artist_id','genres']], on='artist_id', how='left')
    merged['genres'] = merged['genres'].apply(
        lambda g: [x.strip().lower() for x in g if x.strip() and x.strip() != '']
        if isinstance(g, list) else []
    )
    all_genres = merged['genres'].explode().dropna()
    all_genres = all_genres[all_genres != '']
    genres_df = all_genres.value_counts().head(15).reset_index()
    genres_df.columns = ['genre', 'plays']

print(genres_df)

fig, ax = plt.subplots(figsize=(10, 7))
palette = [SPOTIFY_GREEN if i == 0 else '#2d6a4f' for i in range(len(genres_df))]
ax.barh(
    genres_df['genre'][::-1],
    genres_df['plays'][::-1],
    color=palette[::-1],
    edgecolor='none',
    height=0.7,
)
for i, (plays, genre) in enumerate(zip(genres_df['plays'][::-1], genres_df['genre'][::-1])):
    ax.text(plays + 0.3, i, str(int(plays)), va='center', ha='left', fontsize=9, color='white')
ax.set_xlabel('Reproducciones', fontsize=11)
ax.set_title('Top 15 Géneros Más Escuchados', fontsize=14, fontweight='bold', pad=15)
ax.grid(axis='x')
ax.spines[['top','right','left','bottom']].set_visible(False)
plt.tight_layout()
plt.savefig('p4_genres.png', dpi=150, bbox_inches='tight')
plt.show()

**Interpretación:**  
El género dominante es la música cristiana/worship en español, resultado esperado dado el perfil de escucha. Es interesante ver géneros como reggaeton y pop latino en posiciones intermedias, lo que confirma que no toda la escucha es de un solo género. Géneros que sorprenden son algunos tags muy específicos de Last.fm (como 'alabanza' o 'adoracion') que evidencian la naturaleza regional y especializada de los artistas más reproducidos.

---
## Paso 4 — Conclusiones Individuales

### Jhonatan Vera

Antes de hacer este análisis sabía que escuchaba mucha música cristiana, pero no dimensionaba cuánto. Ver los datos me mostró que hay artistas que escucho de forma casi automática — no porque los busque activamente, sino porque quedan en la rotación. Lo que no pude responder con el modelo actual es qué canciones son las que más repito en una misma sesión; el historial registra cada reproducción pero no agrupa por sesión, así que no puedo ver si escucho 5 veces seguidas la misma canción o si las reproducciones están distribuidas en días distintos.

### Suley Suárez

El análisis me reveló que mis horarios de escucha están más definidos de lo que pensaba: hay franjas claras que coinciden con mis rutinas diarias. Lo que sí me sorprendió fue la diversidad de géneros secundarios — pensaba que era casi exclusivamente un perfil de un solo género, pero hay más variedad de la que recordaba. Una pregunta que no pude responder es cómo evolucionó mi escucha en el tiempo: si los artistas que escucho hoy son los mismos de hace 6 meses o si hubo un cambio de etapa musical, porque el modelo solo captura los top artistas actuales y el historial reciente de Spotify (últimas 50 reproducciones por sesión).

---
## Checklist de Entrega

- [x] Conexión al DWH usando variables de entorno (`python-dotenv`) — sin credenciales en el código
- [x] `shape`, `dtypes` y % de nulos visibles para las 4 tablas
- [x] `describe()` de columnas numéricas con interpretación escrita debajo
- [x] Pregunta 1: gráfico de barras horizontales + 2–3 oraciones de interpretación
- [x] Pregunta 2: gráfico de barras por hora + 2–3 oraciones de interpretación
- [x] Pregunta 3: estadísticas impresas + histograma + 2–3 oraciones de interpretación
- [x] Pregunta 4: géneros desanidados con `UNNEST` + gráfico + 2–3 oraciones de interpretación
- [x] Párrafo de conclusiones de cada integrante por separado
- [ ] El notebook corre limpio con **Kernel → Restart & Run All** sin errores *(verificar antes de entregar)*